# Test notebook for DPD funcionality

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from flowermd.library import LJChain

molecules = LJChain(
    num_mols=500,
    lengths=100,
    bead_sequence=["_A"],
    bead_mass={"_A": 1.0},
    bond_lengths={"_A-_A": 1.0},
)

# I think my units are off here. In the KremerGrest model reruns, I used Chris's HardSphereRandomWalk
# this code did not require reference units...

## New Random Walk System Class

In [3]:
from flowermd.library import RandomWalk
import unyt as u

ref_length = 1.0 * u.Unit(
    "nm"
)  # not sure if these are correct for reduced units
ref_mass = 1.0 * u.Unit("g/mol")
ref_energy = 1.0 * u.Unit("kcal / mol")
ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}

system = RandomWalk(
    molecules=molecules,
    density=1.0 * u.Unit("m**-3"),
    bond_length=1.0,
    buffer=0.5,
)

(50000, 3)


## FF from GMSO XML file

In [4]:
from flowermd.library.forcefields import Bead_Spring_DPD

In [5]:
system.apply_forcefield(force_field=Bead_Spring_DPD(), r_cut=1.01, kT=1.0)

KeyError: 'epsilon'

## Forcefield class

In [ ]:
from flowermd.library import PhantomWalk

sim = PhantomWalk(
    initial_state=system.hoomd_snapshot,
    forcefield=system.hoomd_forcefield,
    gsd_write_freq=100,
    log_write_freq=50,
    n_steps_dpd=500,
    n_steps_fire=100,
)

In [ ]:
import hoomd

for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
# system.to_gsd("random_walk_test.gsd")